# Week6_ex1 — Single line-to-ground faults on a five-bus system

Reproduces Example 10.8 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a bolted phase-a-to-ground fault at each bus of the Example 8.5 system, solved with the positive-, negative- and zero-sequence bus impedance matrices and checked against PowerWorld's fault analysis (pre-fault profile = solved power flow, all buses at 1.05 pu).

![PowerWorld one-line](Images/Week6_ex1_result_image.png)

In [1]:
import numpy as np
import pandas as pd
import pandapower as pp
from case5 import (S_BASE, V_PREFAULT, BUS_KV, BUSES, GENS, TRAFOS, LINES, sequence_table,
                   build_case, pandapower_ybus, sequence_ybus, slg_fault, to_phase, compare)

SRC = "PowerWorld Fault Analysis (Run Mode)"

# 1. Sequence reactances on S_base; delta-grounded wye transformers with zero phase shift
sequence_table()

,Element,X1 [pu],X2 [pu],X0 [pu],Zero-seq path
0,G1,0.0450,0.0450,0.0125,X0 + 3Xn = 0.0125 to ground at bus 1
1,G3,0.0225,0.0225,0.0050,X0 + 3Xn = 0.0125 to ground at bus 3
2,T 5-1,0.0200,0.0200,0.0200,"X0 to ground at bus 5, open at bus 1 (delta)"
3,T 4-3,0.0100,0.0100,0.0100,"X0 to ground at bus 4, open at bus 3 (delta)"
4,L 4-2,0.1000,0.1000,0.3000,series
5,L 5-2,0.0500,0.0500,0.1500,series
6,L 5-4,0.0250,0.0250,0.0750,series


In [2]:
# 2. Pre-fault state: no load, both generators hold 1.05 pu
net = build_case()
pp.runpp(net, numba=False)

# 3. The hand-built positive-sequence Ybus must equal pandapower's own Ybus (plus 1/jX1),
#    so the same stamping code can be trusted for the zero-sequence network
Y = {s: sequence_ybus(s) for s in (1, 2, 0)}
mismatch = np.abs(Y[1] - pandapower_ybus(net)).max()
assert mismatch < 1e-9, mismatch
Z = {s: np.linalg.inv(y) for s, y in Y.items()}

pd.DataFrame({"Bus": BUSES, "Pre-fault V [pu]": net.res_bus.vm_pu.loc[BUSES].values,
              "Z1 [pu]": [Z[1][k, k].imag for k in range(5)],
              "Z2 [pu]": [Z[2][k, k].imag for k in range(5)],
              "Z0 [pu]": [Z[0][k, k].imag for k in range(5)]}).round(5)

,Bus,Pre-fault V [pu],Z1 [pu],Z2 [pu],Z0 [pu]
0,1,1.05,0.02797,0.02797,0.01250
1,2,1.05,0.05695,0.05695,0.10894
2,3,1.05,0.01824,0.01824,0.01250
3,4,1.05,0.02362,0.02362,0.00894
4,5,1.05,0.02947,0.02947,0.01576


In [3]:
# 4. Fault Definitions (Auto Insert, single bus): Fault 1 = 3PB, Fault 2 = SLG
x1_pw = {1: 0.02797, 2: 0.05695, 3: 0.01824, 4: 0.02362, 5: 0.02947}
x_slg_pw = {1: 0.06845, 2: 0.22284, 3: 0.04899, 4: 0.05618, 5: 0.07471}
i_slg_pw = {1: 46.02171, 2: 14.13549, 3: 64.30344, 4: 56.07314, 5: 42.16499}

# 5. "Fault 2 Thev X" is the series sum Z1 + Z2 + Z0 seen by an SLG fault. Currents keep
#    one displayed digit (1e-5) for the same reason as the balanced study: PowerWorld's
#    pre-fault voltage is a solved value, not exactly 1.05
rows = []
for b in BUSES:
    k = BUSES.index(b)
    i_seq, _ = slg_fault(Z[1], Z[2], Z[0], k)
    rows.append((f"Bus {b} Z1 [pu]", x1_pw[b], abs(Z[1][k, k]), 5e-6, SRC))
    rows.append((f"Bus {b} Z1+Z2+Z0 [pu]", x_slg_pw[b],
                 abs(Z[1][k, k] + Z[2][k, k] + Z[0][k, k]), 5e-6, SRC))
    rows.append((f"Bus {b} I_F = 3 I0 [pu]", i_slg_pw[b], abs(3 * i_seq), 1e-5, SRC))
compare(rows).round(6)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,Bus 1 Z1 [pu],0.02797,0.027973,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
1,Bus 1 Z1+Z2+Z0 [pu],0.06845,0.068446,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
2,Bus 1 I_F = 3 I0 [pu],46.02171,46.021718,0.000008,0.000010,PowerWorld Fault Analysis (Run Mode)
3,Bus 2 Z1 [pu],0.05695,0.056952,0.000002,0.000005,PowerWorld Fault Analysis (Run Mode)
4,Bus 2 Z1+Z2+Z0 [pu],0.22284,0.222843,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
5,Bus 2 I_F = 3 I0 [pu],14.13549,14.135494,0.000004,0.000010,PowerWorld Fault Analysis (Run Mode)
6,Bus 3 Z1 [pu],0.01824,0.018243,0.000003,0.000005,PowerWorld Fault Analysis (Run Mode)
7,Bus 3 Z1+Z2+Z0 [pu],0.04899,0.048986,0.000004,0.000005,PowerWorld Fault Analysis (Run Mode)
8,Bus 3 I_F = 3 I0 [pu],64.30344,64.303448,0.000008,0.000010,PowerWorld Fault Analysis (Run Mode)
9,Bus 4 Z1 [pu],0.02362,0.023619,0.000001,0.000005,PowerWorld Fault Analysis (Run Mode)


In [4]:
# 6. SLG fault at bus 2: phase voltages during the fault (Single Fault > Bus Records)
k2 = BUSES.index(2)
i_seq, V012 = slg_fault(Z[1], Z[2], Z[0], k2)
Vabc = to_phase(V012)

v_pw = {1: (0.88318, 1.01088, 1.01088, -115.90), 2: (0.0, 1.19152, 1.19151, -130.26),
        3: (0.92138, 1.01937, 1.01937, -116.87), 4: (0.84351, 1.01583, 1.01583, -116.47),
        5: (0.75620, 1.01786, 1.01786, -116.70)}

# 7. Phase C mirrors phase B (+angle); the faulted phase-a angle at bus 2 is 0/0 noise and
#    is not compared. Bus 2 B and C differ by one digit in PowerWorld, so 1e-5 again
rows = []
for b, (va, vb, vc, ang_b) in v_pw.items():
    k = BUSES.index(b)
    rows += [(f"Bus {b} Va [pu]", va, abs(Vabc[0, k]), 1e-5, SRC),
             (f"Bus {b} Vb [pu]", vb, abs(Vabc[1, k]), 1e-5, SRC),
             (f"Bus {b} Vc [pu]", vc, abs(Vabc[2, k]), 1e-5, SRC),
             (f"Bus {b} Vb angle [deg]", ang_b, np.angle(Vabc[1, k], deg=True), 5e-3, SRC),
             (f"Bus {b} Vc angle [deg]", -ang_b, np.angle(Vabc[2, k], deg=True), 5e-3, SRC)]
compare(rows).round(5)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,Bus 1 Va [pu],0.88318,0.88318,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
1,Bus 1 Vb [pu],1.01088,1.01088,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
2,Bus 1 Vc [pu],1.01088,1.01088,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
3,Bus 1 Vb angle [deg],-115.90000,-115.90217,0.00217,0.00500,PowerWorld Fault Analysis (Run Mode)
4,Bus 1 Vc angle [deg],115.90000,115.90217,0.00217,0.00500,PowerWorld Fault Analysis (Run Mode)
5,Bus 2 Va [pu],0.00000,0.00000,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
6,Bus 2 Vb [pu],1.19152,1.19151,0.00001,0.00001,PowerWorld Fault Analysis (Run Mode)
7,Bus 2 Vc [pu],1.19151,1.19151,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
8,Bus 2 Vb angle [deg],-130.26000,-130.25566,0.00434,0.00500,PowerWorld Fault Analysis (Run Mode)
9,Bus 2 Vc angle [deg],130.26000,130.25566,0.00434,0.00500,PowerWorld Fault Analysis (Run Mode)


In [5]:
# 8. SLG fault at bus 2: branch phase currents (Single Fault > Lines), both ends
v = dict(zip(BUSES, V012.T))

def seq_current(f, t, x, x0):
    """Sequence currents entering a line at bus f."""
    return np.array([(v[f][0] - v[t][0]) / (1j * x0),
                     (v[f][1] - v[t][1]) / (1j * x),
                     (v[f][2] - v[t][2]) / (1j * x)])

rows = []
# 9. Transformers: From = delta bus, To = grounded wye bus. Zero-sequence current flows
#    only on the wye side (into the neutral), so the delta side carries I1 + I2 alone
i_tr_pw = {(1, 5): (3.70721, 1.85361, 6.34869, 0.78788),
           (3, 4): (5.71646, 2.85823, 7.78681, 0.78788)}
for w, d, x, x0 in TRAFOS:
    delta_end = np.array([0, (v[d][1] - v[w][1]) / (1j * x), (v[d][2] - v[w][2]) / (1j * x)])
    wye_end = np.array([v[w][0] / (1j * x0), (v[w][1] - v[d][1]) / (1j * x),
                        (v[w][2] - v[d][2]) / (1j * x)])
    ia_f, ib_f, ia_t, ib_t = i_tr_pw[(d, w)]
    Id, Iw = np.abs(to_phase(delta_end)), np.abs(to_phase(wye_end))
    rows += [(f"T {d}-{w} Ia from [pu]", ia_f, Id[0], 1e-5, SRC),
             (f"T {d}-{w} Ib from [pu]", ib_f, Id[1], 1e-5, SRC),
             (f"T {d}-{w} Ia to [pu]", ia_t, Iw[0], 1e-5, SRC),
             (f"T {d}-{w} Ib to [pu]", ib_t, Iw[1], 1e-5, SRC)]

# 10. Lines carry the same current at both ends (no shunt charging)
i_ln_pw = {(4, 2): (5.15111, 0.11255), (5, 2): (8.98438, 0.11255), (5, 4): (2.63569, 0.67532)}
for f, t, x, x0 in LINES:
    I = np.abs(to_phase(seq_current(f, t, x, x0)))
    rows += [(f"L {f}-{t} Ia [pu]", i_ln_pw[(f, t)][0], I[0], 1e-5, SRC),
             (f"L {f}-{t} Ib [pu]", i_ln_pw[(f, t)][1], I[1], 1e-5, SRC)]

# 11. Generators: I1 from the 1.05 pu source behind X1, I2 and I0 driven by the bus voltage
i_gen_pw = {1: (3.70721, 1.85361), 3: (5.71646, 2.85823)}
for b, x1, x2, x0, xn in GENS:
    i012 = np.array([-v[b][0] / (1j * (x0 + 3 * xn)), (V_PREFAULT - v[b][1]) / (1j * x1),
                     -v[b][2] / (1j * x2)])
    I = np.abs(to_phase(i012))
    rows += [(f"G{b} Ia [pu]", i_gen_pw[b][0], I[0], 1e-5, SRC),
             (f"G{b} Ib [pu]", i_gen_pw[b][1], I[1], 1e-5, SRC)]
compare(rows).round(5)

,Quantity,PowerWorld,pandapower,Abs diff,Tolerance,Source
0,T 1-5 Ia from [pu],3.70721,3.70721,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
1,T 1-5 Ib from [pu],1.85361,1.85360,0.00001,0.00001,PowerWorld Fault Analysis (Run Mode)
2,T 1-5 Ia to [pu],6.34869,6.34869,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
3,T 1-5 Ib to [pu],0.78788,0.78788,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
4,T 3-4 Ia from [pu],5.71646,5.71646,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
5,T 3-4 Ib from [pu],2.85823,2.85823,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
6,T 3-4 Ia to [pu],7.78681,7.78681,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
7,T 3-4 Ib to [pu],0.78788,0.78788,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
8,L 4-2 Ia [pu],5.15111,5.15111,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)
9,L 4-2 Ib [pu],0.11255,0.11255,0.00000,0.00001,PowerWorld Fault Analysis (Run Mode)


In [6]:
# 12. SLG vs three-phase fault level at every bus (pu and kA, kA base = S_base / (sqrt3 kV))
rows = []
for b in BUSES:
    k = BUSES.index(b)
    i_slg = abs(3 * slg_fault(Z[1], Z[2], Z[0], k)[0])
    i_3ph = V_PREFAULT / abs(Z[1][k, k])
    i_base = S_BASE / (np.sqrt(3) * BUS_KV[b])
    rows.append({"Bus": b, "kV": BUS_KV[b], "Z0 / Z1": abs(Z[0][k, k] / Z[1][k, k]),
                 "I_3ph [pu]": i_3ph, "I_SLG [pu]": i_slg, "SLG / 3ph": i_slg / i_3ph,
                 "I_3ph [kA]": i_3ph * i_base, "I_SLG [kA]": i_slg * i_base})
pd.DataFrame(rows).round(4)

,Bus,kV,Z0 / Z1,I_3ph [pu],I_SLG [pu],SLG / 3ph,I_3ph [kA],I_SLG [kA]
0,1,15.0,0.4469,37.5362,46.0217,1.2261,144.4770,177.1377
1,2,345.0,1.9128,18.4366,14.1355,0.7667,3.0853,2.3655
2,3,15.0,0.6852,57.5556,64.3034,1.1172,221.5314,247.5041
3,4,345.0,0.3785,44.4565,56.0731,1.2613,7.4397,9.3837
4,5,345.0,0.5346,35.6240,42.1650,1.1836,5.9616,7.0562
